# Genesis RNA - BRCA Variant Classifier

Production-ready training pipeline using **real genomic data** (no label leakage).

**Expected accuracy:** 75-85% (realistic for variant prediction)

## Setup

In [ ]:
# Mount Google Drive for saving results
from google.colab import drive
drive.mount('/content/drive')

# Create output directory
!mkdir -p /content/drive/MyDrive/genesis_rna_results

In [ ]:
# Install dependencies
!pip install -q torch transformers scikit-learn pandas numpy biopython requests

In [ ]:
# Clone repository and install
!git clone https://github.com/oluwafemidiakhoa/genesi_ai.git
%cd genesi_ai/genesis_rna
!pip install -q -e .

## Data: Fetch Real BRCA Sequences

This fetches real mRNA from Ensembl and applies variants. **NO label leakage.**

In [ ]:
# Download ClinVar BRCA variants
%cd /content/genesi_ai
!python scripts/download_brca_variants.py --output data/breast_cancer/clinvar_brca.csv

In [ ]:
# Fetch real sequences (takes 5-10 minutes)
!python scripts/fetch_real_brca_sequences.py \
    --clinvar data/breast_cancer/clinvar_brca.csv \
    --output data/breast_cancer/real_sequences.csv

# Check results
import pandas as pd
df = pd.read_csv('data/breast_cancer/real_sequences.csv')
print(f"\nLoaded {len(df)} sequences")
print(f"Real variants: {len(df[df['SequenceType']=='variant'])}")
print(f"Reference fallbacks: {len(df[df['SequenceType']=='reference_fallback'])}")

## Train/Test Split

Using temporal split (train on old, test on new) to prevent memorization.

In [ ]:
!python scripts/proper_train_test_split.py \
    --input data/breast_cancer/real_sequences.csv \
    --train_out data/breast_cancer/train.csv \
    --test_out data/breast_cancer/test.csv \
    --method temporal \
    --split_date 2020-01-01

## Baseline Models First

Test simple k-mer models before using transformers.

In [ ]:
!python scripts/baseline_models.py \
    --train data/breast_cancer/train.csv \
    --test data/breast_cancer/test.csv \
    --output results/baseline_results.csv

# Show results
results = pd.read_csv('results/baseline_results.csv')
print("\nBaseline Results:")
print(results[['model', 'accuracy', 'f1', 'auc_roc']])

## Decision Point

**If baseline >80%:** Use that model (transformers not needed)

**If baseline <80%:** Train Genesis RNA transformer below

## Train Genesis RNA (Only if baseline failed)

In [ ]:
# Check baseline accuracy
best_baseline = results['accuracy'].max()
print(f"Best baseline accuracy: {best_baseline:.1%}")

if best_baseline >= 0.80:
    print("\n✅ Baseline is sufficient! No need for transformers.")
    print("Skipping Genesis RNA training.")
else:
    print(f"\n⚠️ Baseline only {best_baseline:.1%}. Training Genesis RNA...")

In [ ]:
# Train Genesis RNA (only if needed)
if best_baseline < 0.80:
    %cd /content/genesi_ai/genesis_rna
    !python -m genesis_rna.train_pretrain \
        --config ../configs/train_t4_optimized.yaml \
        --data_path ../data/breast_cancer/train.csv \
        --output_dir /content/drive/MyDrive/genesis_rna_results/checkpoints \
        --num_epochs 30
    
    print("\n✅ Training complete! Checkpoints saved to Google Drive.")

## Results

Expected: 75-85% accuracy (realistic for variant prediction)

In [ ]:
# Load and display final results
import glob
import json

# Find metrics file
metrics_files = glob.glob('/content/drive/MyDrive/genesis_rna_results/**/*metrics*.csv', recursive=True)
if metrics_files:
    metrics = pd.read_csv(metrics_files[0])
    print("\nTraining Metrics:")
    print(metrics.tail())
    
    final_acc = metrics['val_accuracy'].iloc[-1]
    print(f"\n📊 Final Validation Accuracy: {final_acc:.1%}")
    
    if final_acc >= 0.95:
        print("\n🚨 WARNING: Accuracy suspiciously high!")
        print("   Check for data leakage!")
    elif final_acc >= 0.80:
        print("\n✅ Excellent performance!")
    elif final_acc >= 0.70:
        print("\n✅ Good performance for variant prediction.")
    else:
        print("\n⚠️ Performance lower than expected.")
        print("   Consider: more data, better features, or different approach.")
else:
    print("No metrics file found. Check if training completed.")

## Download Results

In [ ]:
# Zip results for download
!cd /content/drive/MyDrive/genesis_rna_results && \
    zip -r genesis_rna_results.zip . && \
    mv genesis_rna_results.zip /content/

print("\n✅ Results zipped!")
print("Download: /content/genesis_rna_results.zip")
print("\nOr access directly in Google Drive:")
print("/MyDrive/genesis_rna_results/")

---

**Done!** 

Results saved to Google Drive: `/MyDrive/genesis_rna_results/`

**Expected accuracy:** 75-85% (realistic, no label leakage)